<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">

<br><br><br>
<h2><font color="#00586D" size=5>Capstone XI: Servicios avanzados en la nube</font></h2>



<h1><font color="#00586D" size=6>Cartelera</font></h1>

<br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Juan Ignacio Alonso Barba, Jesus Martínez Gómez, Cristina Romero González</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube V </font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>

</div>

---

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Introducción](#section1)
* [2. Presentación de la arquitectura](#section2)
* [3. Acciones a realizar](#section3)

---

<div class="alert alert-block alert-info"> 
<b>IMPORTANTE</b> Utiliza el espacio de AWS Academy adecuado, en función de la fecha estimada en la que crees que puedes terminar  
 
* Si crees que vas a terminar antes del 1 de Mayo: https://awsacademy.instructure.com//courses/107104
* Si crees que vas a terminar después del 1 de Mayo: https://awsacademy.instructure.com//courses/110625
  
</div>

<div class="alert alert-block alert-warning"> 
<b>MUY IMPORTANTE</b> Presta atención a las restricciones del entorno AWS Academy, o tu cuenta puede ser bloqueada, de forma que pierdas todo el desarrollo realizado hasta el momento.
<br>
<br>
    
En concreto, **no puede haber más de 10 funciones lambda** ejecutándose de forma concurrente

Uno de los escenarios que pueden generar esa situación es el uso de disparadores/triggers, desde S3 o DynamoDB, de forma que si subimos 11 ficheros de una y cada uno tiene una ejecución asociada, se genere esta situación.

Para poder evitar cualquier tipo de problema, seguiremos la siguiente configuración:
* No tener más funciones lambda de las estrictamente necesarias, y nunca más de 9
* Establecer, para todas las funciones, una concurrencia máxima de 1, de forma que no haya más de una ejecución por función
    * Dentro de la cada función lambda --> Configuración --> Concurrencia --> Editar Simultaneidad --> Seleccionar 1
  
</div>

In [ ]:
import os
session = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=os.environ.get("AWS_SESSION_TOKEN")  # optional for temporary creds,
    region_name='us-east-1'
)

dynamodb = session.resource('dynamodb')

sts_client = session.client('sts')
identity = sts_client.get_caller_identity()
print(identity)

<a id="section1"></a>
# <font color="#00586D"> 1. Introducción</font>

El objetivo de este capstone es combinar de forma efectiva algunos de los conocimientos obtenidos en el módulo 11. 

El Capstone posee un enfoque decididamente práctico, de forma que su resolución permitirá a los/as alumnos/as desplegar una pequeña aplicación en la nube que, de manera recurrente, consulte el estado de las carteleras de cine a través de llamadas a un API de terceros. El resultado de dichas consultas será almacenado a través de los servicios de almacenamiento de AWS. Además, los datos más relevantes serán almacenados en una base de datos no relacional, la cual permitirá realizar futuras consultas a través de un API construida sobre funciones Lambda. 

La validación de la aplicación desarrollada se podrá realizar desde la libreta actual. 

Cualquier alumno/a puede sentirse libre de ampliar los objetivos del Capstone.

<div style="text-align: right">
<a href="#inicio"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section2"></a> 
# <font color="#00586D"> 2. Presentación de la arquitectura</font>

Durante el Capstone se desarrollará y desplegará una aplicación con la siguiente funcionalidad:
<ol type="A">
    <li> Consultará datos a través de un API externa, que permite obtener información sobre las películas que actualmente se encuentran proyectadas en los cines. Inicialmente, las consultas se harán a través del uso de la librería request dentro de la libreta actual</li>
    <li> Almacenará los datos obtenidos con cada llamada, en forma de ficheros .json, usando el servicio de almacenamiento AWS S3. Estos ficheros se organizarán a través de un nombrado que refleje la fecha en la que han sido obtenidos</li>
    <li> Mantendrá actualizada una base de datos, creada en AWS DynamoDB, con una serie de datos clave para toda película de la que hayamos obtenido información. El proceso de actualización se iniciará ante la presencia de nuevos ficheros en AWS S3.
    <li> Ofrecerá consultas a la base de datos que permitan obtener películas que cumplan determinadas condiciones </li>
        <ol>
            <li> Películas lanzadas en mes en concreto </li>
            <li> Películas con una valoración por encima de un umbral</li>
        </ol>    
    <li> Automatizará las consultas a la API externa a través del despliegue de una función AWS Lambda</li>    
    <li> Ofrecerá las consultas a la base de datos a través de un API desplegado con API Gateway + AWS Lambda</li>    
</ol>

La arquitectura de la aplicación se muestra en la siguiente imagen: 
  
<img src="img/arquitectura.png" alt="Arquitectura del capstone">

<div style="text-align: right">
<a href="#inicio"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<a id="section3"></a> 
# <font color="#00586D" size=5> 3. Acciones a realizar </font>

En esta sección se plantean una serie de ejercicios. Cada ejercicio amplía, de forma incremental, la funcionalidad de la aplicación.

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 0:</b></font> Uso del API "The Movie Database API"

Tal y como se ha visto en otros módulos, para consumir el API debemos obtener un `API Key` que nos identifique durante nuestras consultas.

La API de TMDb requiere autentificación, por lo que para trabajar con ella es necesario, en primer lugar, disponer de un usuario. Una vez hecho el registro en el sitio, es necesario solicitar una clave para el uso de la API. Las instrucciones detalladas se muestran en esta página ([enlace](https://www.themoviedb.org/settings/api)). Este proceso es sencillo, y básicamente consiste en 4 pasos:

- Entrar en la configuración de la cuenta personal.
- Entrar en el menú de la API.
- Crear la API (hay que rellenar un formulario con información de la aplicación)
  * Tipo de aplicación: Educación
  * URL: https://www.cidaen.es/
  * El resto de campos son obligatorios pero se puede rellenar con cualquier contenido
- Identificar la clave de la API (lo usaremos como `API KEY`)

 
Una vez obtenido el API, podemos utilizarlo para realizar cualquier consulta de las incluidas en la documentación. Por ejemplo, se pueden realizar peticiones del tipo https://api.themoviedb.org/3/movie/550?api_key=TU_CLAVE

Durante el capstone, haremos uso de la llamada now_playing, la cual nos da los siguientes datos para cada película proyectada actualmente en cines:

- poster_path: string con valores para generar una URL al poster de la película
- adult: valor boolean que nos indica si es una película para adultos o no
- overview: string con el resumen de la película
- release_date: string con la fecha de lanzamiento
- id: número entero que sirve de identificador único
- popularity: número entero con la popularidad actual
- vote_average: número entero con el valor medio de los votos realizados hasta la fecha
- vote_count: número entero que indica el número de votos realizados
- otros campos que podemos consultar desde https://api.themoviedb.org/3/movie/now_playing



<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

Utilizando alguna herramienta tipo POSTMAN, realiza una llamada a la URL https://api.themoviedb.org/3/movie/now_playing?api_key=TU_CLAVE reemplazando TU_CLAVE por el API Key personal. 

Pegad los datos obtenidos para las 3 primeras películas en la siguiente celda:

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 1:</b></font> Consulta a través de requests

- Rellenad la celda de código a continuación para realizar una consulta a través del uso de la librería requests
  -  Incluid el valor del API KEY previamente obtenido
- Sabiendo que los datos se obtienen ordenados por popularidad, añadid un valor a cada resultado llamado rank con la posición de la película
   - La película en la posición 0 del array tendrá el valor de ranking 1
   - La película en la posición 1 del array tendrá el valor de ranking 2
   - La película en la posición n-1 del array tendrá el valor de ranking n

In [64]:
import os
import requests

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
response = requests.get(f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1")

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1

    # Mostrar las 3 primeras películas con el ranking añadido
    for pelicula in peliculas[:3]:
        print(f"{pelicula['rank']}. {pelicula['title']} - Popularidad: {pelicula['popularity']}")
else:
    print(f"Error en la consulta: {response.status_code}")

1. Una película de Minecraft - Popularidad: 634.0126
2. Lilo y Stitch - Popularidad: 719.8697
3. Destino final: Lazos de sangre - Popularidad: 488.19


<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

Utilizando el código anterior, pegad los datos obtenidos para las 3 primeras películas (includo el campo rank) en la siguiente celda:

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#113D68"></i>
 </font></div>

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 2:</b></font> Subida de datos a S3

- Cread un bucket de S3 con el nombre `mcidaen5.capstone11.movies.raw.data.<tu-nombre>`.   
  - Lo haremos desde la consola de AWS
- Rellenad la celda de código a continuación para que, tras consultar datos al API de películas e incluir el campo rank, se genere un objeto por cada resultado
- Path del objeto a crear /movies/ID_MOVIE/AÑO_MES_DIA.json
  - ID_MOVIE se corresponde con el id de la película
  - AÑO/MES/DÍA con la fecha de la consulta
  
 Por ejemplo, si la consulta obtiene 4 películas, la ejecución de la celda debe generar 4 ficheros .json que se subirán, usando para ello la librería boto3, al bucket de S3 como objetos

In [65]:
import os
import requests
import json
import boto3    
from datetime import datetime

NOMBRE_BUCKET = os.environ.get("S3_BUCKET_NAME", "<YOUR_BUCKET_NAME>")# Creado desde la consola de AWS 
now = datetime.now()
year_month_day = now.strftime("%Y_%m_%d") # Obtener dígitos del año, mes y día


# 1.- OBTENER DATOS DEL API MOVIES DB (COPIAD DEL EJERCICIO 1)

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
response = requests.get(f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1")

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1

    # Mostrar las 3 primeras películas con el ranking añadido
    for pelicula in peliculas[:3]:
        print(f"{pelicula['rank']}. {pelicula['title']} - Popularidad: {pelicula['popularity']}")
else:
    print(f"Error en la consulta: {response.status_code}")
    
# 2.- CONFIGURAD EL CLIENTE Y RECURSO DE ACCESO A S3, Y VALIDAD SU USO (CREACIÓN DE CARPETA)
    
s3_client = session.client('s3', region_name='us-east-1')
s3_resource = session.resource('s3', region_name='us-east-1')

try:
    s3_client.put_object(Bucket=NOMBRE_BUCKET, Key='movies/')
    print('Carpeta creada correctamente')
except Exception as e:
    print(e) 


# 3.- SUBID A S3 UN OBJETO POR RESULTADO OBTENIDO EN EL PASO 1

for pelicula in peliculas:
    movie_id = pelicula['id']
    ruta_s3 = f"movies/{movie_id}/{year_month_day}.json"
    contenido_json = json.dumps(pelicula, ensure_ascii=False)

    try:
        s3_client.put_object(
            Bucket=NOMBRE_BUCKET,
            Key=ruta_s3,
            Body=contenido_json.encode('utf-8'),
            ContentType='application/json'
        )
        print(f"{ruta_s3} subido correctamente.")
    except Exception as e:
        print(f"Error al subir {ruta_s3}: {e}")

1. Una película de Minecraft - Popularidad: 634.0126
2. Lilo y Stitch - Popularidad: 719.8697
3. Destino final: Lazos de sangre - Popularidad: 488.19
Carpeta creada correctamente
movies/950387/2025_05_27.json subido correctamente.
movies/552524/2025_05_27.json subido correctamente.
movies/574475/2025_05_27.json subido correctamente.
movies/1232546/2025_05_27.json subido correctamente.
movies/575265/2025_05_27.json subido correctamente.
movies/896536/2025_05_27.json subido correctamente.
movies/447273/2025_05_27.json subido correctamente.
movies/1241436/2025_05_27.json subido correctamente.
movies/1098006/2025_05_27.json subido correctamente.
movies/1001414/2025_05_27.json subido correctamente.
movies/977294/2025_05_27.json subido correctamente.
movies/1297028/2025_05_27.json subido correctamente.
movies/986056/2025_05_27.json subido correctamente.
movies/1233069/2025_05_27.json subido correctamente.
movies/1094473/2025_05_27.json subido correctamente.
movies/1144430/2025_05_27.json sub

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

- Ejecutad el código anterior y comprobad que se han generado entradas en S3
- Generad una imagen, con un pantallazo que muestre el contenido de la carpeta movies en S3, y guardadla en la ruta (junto al notebook) `/img/eje_2.png`

<img src="img/eje_2.png" alt="Validacion 2">

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 3:</b></font> Almacenamiento de datos en DynamoDB

Para facilitar futuras consultas, vamos a almacenar datos de las películas en una base de datos de DynamoDB. Para ello, crearemos una tabla de DynamoDB a través de la consola de AWS con la siguiente configuración:

- Nombre: MoviesDB
- Partition Key: id (string)
- Global Secondary Index
  - Partition Key: y_m (string)
     - Representa la fecha de lanzamiento de la película (release_date)
     - Compuesto por el año (4 dígitos) y mes (2 dígitos). Por ejemplo 2025_01
  - Sort Key: val (número)
     - Representa la valoración de la película (vote_average)         

Para validar la tabla creada, vamos a generar una entrada a través de boto3. Para ello, rellenad la celda de código 3.1 de forma que el primer resultado consultado desde el API pueda guardarse en DynamoDB

In [66]:
import os
# CELDA 3.1

import requests
import json
import boto3    
from decimal import Decimal
from datetime import datetime

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
NOMBRE_TABLA = "MoviesDB"

# 1.- OBTENER DATOS DEL API MOVIES DB
response = requests.get(f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1")

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1

    # Mostrar las 3 primeras películas con el ranking añadido
    for pelicula in peliculas[:3]:
        print(f"{pelicula['rank']}. {pelicula['title']} - Popularidad: {pelicula['popularity']}")
else:
    print(f"Error en la consulta: {response.status_code}")
    peliculas = []

# 2.- CONFIGURAD EL RECURSO Y TABLA DE DYNAMO_DB
dynamodb_resource = session.resource('dynamodb', region_name='us-east-1')
dynamodb_table = dynamodb_resource.Table(NOMBRE_TABLA)

# 3.- MODIFICAD FORMATO PARA DINAMODB
if peliculas:
    first_movie = peliculas[0]
    
    first_movie["id"] = str(first_movie["id"])
    
    release_date = first_movie.get("release_date", "0000-00-00")
    try:
        fecha = datetime.strptime(release_date, "%Y-%m-%d")
        y_m = fecha.strftime("%Y_%m")
    except ValueError:
        y_m = "unknown"
        
    first_movie["y_m"] = y_m
    first_movie["val"] = first_movie.get("vote_average", 0)

    # 4.- GUARDAR EN DYNAMO DB
    dynamodb_table.put_item(Item=json.loads(json.dumps(first_movie), parse_float=Decimal))
    print("✅ Película insertada correctamente en DynamoDB.")


1. Una película de Minecraft - Popularidad: 634.0126
2. Lilo y Stitch - Popularidad: 719.8697
3. Destino final: Lazos de sangre - Popularidad: 488.19
✅ Película insertada correctamente en DynamoDB.


<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

- Ejecutad el código anterior y comprobad que se ha generado una entrada en DynamoDB
- Generad una imagen, con un pantallazo que muestre el contenido de la tabla movies en DynamoDB, y guardadla en la ruta (junto al notebook) `/img/eje_3.png`

<img src="img/eje_3.png" alt="Validacion 3">

<div class="alert alert-block alert-info">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i>
    <b>NOTA:</b> Puede que necesitemos cambiar el tipo de algunos campos para que puedan utilizarse como claves de partición u ordenación.
</div>

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#113D68"></i>
 </font></div>

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 4:</b></font> Automatización del proceso

Queremos automatizar el proceso de añadir/actualizar los datos en nuestra tabla de DynamoDB. Para ello, usaremos una función lambda con las siguientes características:

- Debe ejecutarse tras la creación de cualquier objeto en el bucket de S3 creado con anterioridad, bajo la ruta movies/
  - Lo conseguiremos a través de un disparador/trigger
- En caso de que no exista ninguna entrada previa para la película, se creará una entrada una vez se hayan generado campos adecuados para las claves de partición y ordenación
  - Clave de partición: id
  - GSI
      - Clave de partición: y_m
      - Clave de ordenación: val
      
Además, tendrá dos campos "especiales" con las siguientes características:
  - rank_hist
    - Representará un diccionario con la evolución de la posición de la película en los rankings a lo largo de las semanas
    - Para ello, tendrá una entrada por cada semana en la que se haya consultado con el valor obtenido, donde fecha tendrá el formato año_nSemana (número de la semana/año en la que se ha consultado el ranking). 
      - Año estará representado por 4 dígitos
      - nSemana estará representado por 2 dígitos 
    - Si realizamos una petición a fecha 12 de Enero de 2025, estaremos en la semana 2, mientras que el 13 estaremos en la semana 3 (el 1 de Enero fue miércoles)
    - Para la película más popular, añadiremos la entrada
        - rank_hist = {"2025_02", 1}
  - until_date
    - Nos ayudará para consultar el número de días que una película se ha estado proyectando
    - Guardará la fecha de la consulta en el mismo formato que el campo release_date (proporcionado por el API MoviesDB)
        - yyyy-mm-dd
        
- En caso de que ya exista una entrada, actualizaremos sus valores de forma que "podamos" tener los siguientes cambios
    - Actualización de la valoración, de forma que se use el último valor proporcionado por el API de TMDB 
      - Campo val
    - Actualización del campo until_date
    - Nueva entrada en el campo rank_hist con el valor de la semana actual 

Para ello, se deben realizar las siguientes acciones:

- Desarrollad en la celda de código 4.1 el código que se encargue de crear/actualizar entradas en DynamoDB, usando datos obtenidos previamente
- En la consola de AWS, cread una función lambda que se ejecute ante las acciones de S3 indicadas previamente
    - Incluir un disparador de S3 ante la creación de cualquier objeto
- Desplegad el código desarrollado en la celda 4.1 en la función lambda anterior
- Modificad el código para que los datos se obtengan desde el evento de la lambda
  - El evento incluye el Bucket y clave del objeto de S3 disparó la ejecución (fichero .json)
  - Utilizad boto3 para leer el contenido del fichero
  - La celda 4.2 incluye algún código que pueda servir de ayuda
- Desplegad la función lambda tras los cambios
- Ejecutad alguna celda anterior que genere entradas en S3 a partir de llamadas al API de películas y comprobad el resultado
  - También podemos bajar y subir alguno de los .json que ya estén en el bcuket, usando la consola de AWS

<div class="alert alert-block alert-info">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true">
    </i>
    <b>NOTA</b> Se valorará positivamente el uso de la función 'update_item' de boto3 con una 'update expression' que actualice los valores que pueden cambiar sin necesidad de hacer una 'query' previa de consulta.
</div>

In [67]:
import os
import requests
import json
import boto3
from decimal import Decimal
from datetime import datetime

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
NOMBRE_TABLA = "MoviesDB"

# 1.- OBTENER DATOS DEL API MOVIES DB
response = requests.get(
    f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1"
)

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1
else:
    print(f"Error en la consulta: {response.status_code}")
    peliculas = []

# 2.- CONFIGURAR EL RECURSO Y TABLA DE DYNAMODB
dynamodb_resource = session.resource('dynamodb', region_name='us-east-1')
dynamodb_table = dynamodb_resource.Table(NOMBRE_TABLA)

# 3.- MODIFICAR FORMATO PARA DINAMODB Y GUARDAR/ACTUALIZAR SOLO LA PRIMERA PELÍCULA
if peliculas:
    now = datetime.now()
    today_str = now.strftime("%Y-%m-%d")
    year, week, _ = now.isocalendar()
    week_str = f"w{year}_{week:02d}"

    first_movie = peliculas[0]
    movie_id = str(first_movie["id"])
    release_date = first_movie.get("release_date", "0000-00-00")

    try:
        fecha = datetime.strptime(release_date, "%Y-%m-%d")
        y_m = fecha.strftime("%Y_%m")
    except ValueError:
        y_m = "unknown"

    val = Decimal(str(first_movie.get("vote_average", 0)))
    rank = int(first_movie.get("rank", 0))

    # 4.- LÓGICA DE CREACIÓN O ACTUALIZACIÓN DE ENTRADA
    # Comprobar si ya existe el campo rank_hist
    item = dynamodb_table.get_item(Key={"id": movie_id}).get('Item', {})

    if 'rank_hist' not in item:
        # Si no existe, crea el mapa rank_hist con la semana actual
        rank_hist = {week_str: rank}
        update_expr = (
            "SET y_m = :y_m, val = :val, until_date = :until_date, rank_hist = :rank_hist"
        )
        expr_attr_vals = {
            ":y_m": y_m,
            ":val": val,
            ":until_date": today_str,
            ":rank_hist": rank_hist
        }
        dynamodb_table.update_item(
            Key={"id": movie_id},
            UpdateExpression=update_expr,
            ExpressionAttributeValues=expr_attr_vals
        )
    else:
        # Si existe, actualiza solo el campo de la semana en rank_hist
        update_expr = (
            "SET y_m = :y_m, val = :val, until_date = :until_date, rank_hist.#week = :rank"
        )
        expr_attr_vals = {
            ":y_m": y_m,
            ":val": val,
            ":until_date": today_str,
            ":rank": rank
        }
        expr_attr_names = {
            "#week": week_str
        }
        dynamodb_table.update_item(
            Key={"id": movie_id},
            UpdateExpression=update_expr,
            ExpressionAttributeValues=expr_attr_vals,
            ExpressionAttributeNames=expr_attr_names
        )

    print(f"✅ Película {first_movie['title']} creada/actualizada correctamente en DynamoDB.")

✅ Película Una película de Minecraft creada/actualizada correctamente en DynamoDB.


<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

- Comprobad que al subir un fichero a S3 con datos de una película se ejecuta la lambda que acabamos de desplegar
- Comprobad que la función lambda se ejecuta sin errores
- Forzad la generación de ficheros en S3 a través de la ejecución de celdas anteriores
- Generad una imagen, con un pantallazo que muestre la monitorización de la función lambda durante los últimos 15 minutos, y guardadla en la ruta (junto al notebook) `/img/eje_4_a.png`
- Generad una imagen, con un pantallazo que muestre las entradas en la tabla de DynamoDB, y guardadla en la ruta (junto al notebook) `/img/eje_4_b.png`
- Pegad el contenido final de la lambda en la celda 4.3

In [ ]:
# CELDA 4.3
import boto3
import json
from decimal import Decimal
from datetime import datetime


def convert_floats_to_decimal(obj):
    if isinstance(obj, list):
        return [convert_floats_to_decimal(x) for x in obj]
    elif isinstance(obj, dict):
        return {k: convert_floats_to_decimal(v) for k, v in obj.items()}
    elif isinstance(obj, float):
        return Decimal(str(obj))
    else:
        return obj


def lambda_handler(event, context):
    s3_resource = boto3.resource('s3', region_name='us-east-1')
    now = datetime.now()
    year_number = now.isocalendar()[0]
    week_number = now.isocalendar()[1]
    week_str = f"w{year_number}_{week_number:02d}"
    today_str = now.strftime("%Y-%m-%d")
    
    NOMBRE_TABLA = "MoviesDB"
    dynamodb_resource = boto3.resource('dynamodb', region_name='us-east-1')
    dynamodb_table = dynamodb_resource.Table(NOMBRE_TABLA)
    
    for record in event.get('Records', []):
        bucket = record["s3"]["bucket"]["name"]
        key = record["s3"]["object"]["key"]
        obj = s3_resource.Object(bucket, key)
        element = json.load(obj.get()['Body'], parse_float=Decimal)
        
        peliculas = element if isinstance(element, list) else [element]
        
        for pelicula in peliculas:
            movie_id = str(pelicula["id"])
            release_date = pelicula.get("release_date", "0000-00-00")
            try:
                fecha = datetime.strptime(release_date, "%Y-%m-%d")
                y_m = fecha.strftime("%Y_%m")
            except ValueError:
                y_m = "unknown"
            val = Decimal(str(pelicula.get("vote_average", 0)))
            rank = int(pelicula.get("rank", 0))

            item = dynamodb_table.get_item(Key={"id": movie_id}).get('Item', {})

            if not item:
                # Es nuevo: guarda TODO el objeto + especiales
                pelicula['id'] = movie_id
                pelicula['y_m'] = y_m
                pelicula['val'] = val
                pelicula['until_date'] = today_str
                pelicula['rank_hist'] = {week_str: rank}
                item_to_save = convert_floats_to_decimal(pelicula)
                dynamodb_table.put_item(Item=item_to_save)
            else:
                # Ya existe: solo actualiza los campos especiales
                update_expr = (
                    "SET y_m = :y_m, val = :val, until_date = :until_date, rank_hist.#week = :rank"
                )
                expr_attr_vals = {
                    ":y_m": y_m,
                    ":val": val,
                    ":until_date": today_str,
                    ":rank": rank
                }
                expr_attr_names = {
                    "#week": week_str
                }
                dynamodb_table.update_item(
                    Key={"id": movie_id},
                    UpdateExpression=update_expr,
                    ExpressionAttributeValues=expr_attr_vals,
                    ExpressionAttributeNames=expr_attr_names
                )
    return {
        'statusCode': 200,
        'body': 'Películas procesadas correctamente'
    }


<img src="img/eje_4_a.png" alt="Validacion 4a">

<img src="img/eje_4_b.png" alt="Validacion 4b">

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#113D68"></i>
 </font></div>

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 5:</b></font> Automatización de la captura de datos

Una vez tenemos todas las piezas del proceso, queremos proceder a automatizar la captura de datos, lo cual realizaremos a través de una función lambda desplegada en la nube.

Esta función lambda tendrá un código similar al encargado de consultar el API de películas y generar nuevos objetos en S3. 
Una vez desplegada, nos encargaremos de añadir un _trigger_ que la ejecute cada 24 horas.

Para que la función tenga un comportamiento correcto, necesitaremos que maneje dependencias adicionales para usar la librería _requests_, lo cual puede realizarse a través de una _layer_. Además, puede que necesitemos modificar los parámetros por defecto (timeout, memoria, ...)

Se deben realizar las siguientes acciones:

- Desplegad una función lambda, similar a la utilizada en el ejercicio 2, que consulte el API de películas y genere como resultado objetos .json en S3
- Resolved los problemas que permitan utilizar la librería _requests_ a través del uso de la _layer_ 
- Modificad los parámetros de la lambda timeout/memoria en caso que de problemas de ejecución
- Añadir un _trigger_ para que la función se ejecute cada día

### Ejemplo de generación de la layer en linux/mac (revisad en Windows)

```
$ virtualenv --python="/usr/bin/python3.12" .venv
$ source .venv/bin/activate
$ mkdir python
$ cd python/
$ pip install requests "urllib3<2" -t .
$ cd ..
$ zip -r python_modules.zip python/
```

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

- Tras todos los pasos, comprobad que la función se ejecuta sin problemas
- Modificad el trigger para que la función se ejecute cada minuto y esperad 5 minutos
- Generad una imagen con un pantallazo que muestre la monitorización de la función lambda durante los últimos 5 minutos, y guardadla en la ruta (junto al notebook) `/img/eje_5.png`
- Modificad el trigger para que la función se ejecute cada día

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#113D68"></i>
 </font></div>

<img src="img/eje_5_sol.png" alt="Validacion 5">

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 6:</b></font> API listado películas (dado mes y año)

Durante este ejercicio, desarrollaremos una función lambda ejecutada tras la invocación de un endpoint de API Gateway. 

Para ello, generaremos una lambda que a partir de eventos con el siguiente formato: `{'pathParameters': {'month': '1', 'year': '2025'}}`, genere un listado adecuado. Dicho listado se obtiene a partir de una consulta a la base de datos DynamoDB usando las siguientes características:
- Consulta de tipo _query_
- Consulta realizada sobre el índice secundario global
  - Se debe proporcionar un valor para el campo y_m para obtener los datos correctos
- Ordenación descendente, de forma que se obtengan en las primeras posiciones las películas con mayor valoración
- Limitar la consulta a 10 películas

Para que el resultado (listado de películas) pueda interpretarse correctamente por API Gateway, lo incluiremos como json en la respuesta de la lambda, en concreto en el campo `body` (comprobad el contenido del código de la celda 6.1). 

Una vez desarrollada la lambda, la podemos validar en local realizando la siguiente llamada:

`lambda_handler({'pathParameters': {'month': '2', 'year': '2025'}}, None)`

Tras la validación de la lambda, desplegaremos un API a través de API Gateway que permita su invocación. Para ello, definiremos 1 recurso base llamado _/list_ sobre el cual añadiremos 1 subrecurso _/list/{year}_ y un (sub)subrecurso _/list/{year}/{month}_. Sobre el último subrecurso (_{month}_) añadiremos un método GET, el cual conectará las llamadas realizadas sobre `API_URL/stage/list/{year}/{month}` con la lambda anterior, donde _{year}_ y _{month}_ serán reemplazados por valores reales.

El evento recibido por la lambda tendrá el formato previamente definido

`{'pathParameters': {'month': 'MONTH', 'year': 'YEAR'}}`

Se deben realizar las siguientes acciones:

- Desarrollad una lambda a partir de la siguiente celda de código, de forma que al ejecutarla con el evento previamente definido, realice una consulta a DynamoDB
- Validad la lambda desplegándola en AWS e invocándola con un evento compatible
- Desplegad un API en API Gateway (API REST no privado) con los recursos/métodos ya definidos
  - Arbol de recursos /list/{year}/{month}
  - Método: GET sobre {month}, invocando la lambda previamente desplegada, y activando la opción _lambda proxy integration_
- Validad el API de manera preliminar realizando llamadas a través de herramientas tipo POSTMAN
    - Posteriormente validaremos el API al completo

In [78]:
import boto3
import decimal
import json
from boto3.dynamodb.conditions import Key

NOMBRE_TABLA = "MoviesDB"
NOMBRE_GSI = "y_m-val-index"  # Cambia si tu GSI tiene otro nombre

dynamo_resource = session.resource('dynamodb', region_name='us-east-1')
dynamo_table = dynamo_resource.Table(NOMBRE_TABLA)

class DecimalEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, decimal.Decimal):
            return float(o)
        return super(DecimalEncoder, self).default(o)
    
def lambda_handler(event, context):
    year = int(event['pathParameters']['year'])
    month = int(event['pathParameters']['month'])
    y_m = f'{year}_{month:02d}'
    
    # Consulta sobre el GSI (y_m como partition key, val como sort key descendente)
    response = dynamo_table.query(
        IndexName=NOMBRE_GSI,
        KeyConditionExpression=Key('y_m').eq(y_m),
        ScanIndexForward=False,    # Orden descendente (mayor valoración primero)
        Limit=10                   # Solo los 10 primeros
    )
    items = response.get('Items', [])
    
    return {
        'statusCode': 200,
        'body': json.dumps(items, cls=DecimalEncoder)
    }

In [80]:
# VALIDACIÓN 

event = {'pathParameters': {'month': '2', 'year': '2025'}} # Cambiad el mes si fuera necesario
lambda_handler(event, {})

{'statusCode': 200,
 'body': '[{"original_title": "In the Lost Lands", "genre_ids": [28.0, 14.0, 12.0], "val": 6.407, "adult": false, "y_m": "2025_02", "overview": "Basada en el relato de George R. R. Martin. Una reina (Amara Okereke), desesperada por encontrar la felicidad en el amor, env\\u00eda a la poderosa bruja Gray Alys (Milla Jovovich) a las Tierras Perdidas, en busca de un poder m\\u00e1gico que permite a una persona transformarse en un hombre lobo. Con el misterioso cazador Boyce (Dave Bautista), que la apoya en la lucha contra criaturas oscuras y despiadadas, Gray deambula por un mundo inquietante y peligroso. Pero solo ella sabe que, cada deseo que se concede, tiene consecuencias inimaginables.", "vote_average": 6.407, "rank": 17.0, "rank_hist": {"w2025_22": 17.0}, "popularity": 142.6165, "backdrop_path": "/op3qmNhvwEvyT7UFyPbIfQmKriB.jpg", "release_date": "2025-02-27", "original_language": "en", "vote_count": 391.0, "until_date": "2025-05-27", "id": "324544", "poster_path"

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 7:</b></font> API detalles película (dado su id)

Durante este ejercicio, desarrollaremos otra función lambda ejecutada tras la invocación de otro endpoint de API Gateway. 

Para ello, generaremos una lambda (desarrollada inicialmente en la celda 7.1) que a partir de eventos con el siguiente formato: `{'pathParameters': {'id': '1300607'}}`, obtenga detalles para la película con el id proporcionado. Los detalles se obtienen a partir de una consulta a la base de datos DynamoDB usando las siguientes características:
- Consulta de tipo _get_item()_
- Consulta realizada sobre la tabla (usando la clave de partición)
  - Se debe proporcionar un valor para el campo id

Para que el resultado (detalles de la película) pueda interpretarse correctamente por API Gateway, lo incluiremos como json en la respuesta de la lambda, en concreto en el campo `body` (comprobad el contenido del código incluido). 

Una vez desarrollada la lambda, la podemos validar en local realizando la siguiente llamada:

`lambda_handler({'pathParameters': {'id': '1300607'}}, None)`

Tras la validación de la lambda, desplegaremos un API a través de API Gateway que permita su invocación. Para ello, definiremos 1 nuevo recurso base llamado _/movies_ sobre el cual añadiremos 1 subrecurso _/movies/{id}_. Sobre el subrecurso (_{id}_) añadiremos un método GET, el cual conectará las llamadas realizadas sobre `API_URL/stage/movies/{id}` con la lambda anterior, donde _{id}_ será reemplazado por un valor real.

El evento recibido por la lambda tendrá el formato previamente definido

`{'pathParameters': {'id': 'MOVIE_ID'}}`

Se deben realizar las siguientes acciones:

- Desarrollad una lambda a partir de la siguiente celda de código, de forma que al ejecutarla con el evento previamente definido, realice una consulta a DynamoDB
- Validad la lambda desplegándola en AWS e invocándola con un evento compatible
- Cread los nuevos recursos/métodos sobre el API de API Gateway previamente desplegado
  - Arbol de recursos /movies/{id}
  - Método: GET sobre {id}, invocando la lambda previamente desplegada, y activando la opción _lambda proxy integration_
- Desplegad de nuevo el API usando la opción implementar API
- Validad el API de manera preliminar realizando llamadas a través de herramientas tipo POSTMAN
    - Posteriormente validaremos el API al completo

In [83]:
# CELDA 7.1

import boto3
import decimal
import json
from boto3.dynamodb.conditions import Key

NOMBRE_TABLA = "MoviesDB"
dynamo_resource = session.resource('dynamodb', region_name='us-east-1')
dynamo_table = dynamo_resource.Table(NOMBRE_TABLA)


class DecimalEncoder(json.JSONEncoder):       # Necesario para manejar los tipos Decimal
    def default(self, o):
        if isinstance(o, decimal.Decimal):
            return str(o)
        return super(DecimalEncoder, self).default(o)
    
    
def lambda_handler(event, context):
    movie_id = event['pathParameters']['id']
    response = dynamo_table.get_item(Key={'id': movie_id})
    item = response.get('Item')
    if item:
        return {
            'statusCode': 200,
            'body': json.dumps(item, cls=DecimalEncoder)
        }
    else:
        return {
            'statusCode': 404,
            'body': json.dumps({'message': f'No se encontro pelicula con id {movie_id}'})
        }

In [85]:
# VALIDACIÓN 

event = {'pathParameters': {'id': '1241436'}}
lambda_handler(event, {})

{'statusCode': 200,
 'body': '{"original_title": "Warfare", "genre_ids": ["10752", "28"], "val": "7.241", "adult": false, "y_m": "2025_04", "overview": "Basada en las experiencias reales del ex marine Ray Mendoza (codirector y coguionista de la pel\\u00edcula) durante la guerra de Irak. Introduce al espectador en la experiencia de un pelot\\u00f3n de Navy SEALs estadounidenses. Concretamente en una misi\\u00f3n de vigilancia que se tuerce en territorio insurgente. Una historia visceral y a pie de campo sobre la guerra moderna y la hermandad, contada como nunca antes: en tiempo real y basada en los recuerdos de quienes la vivieron.", "vote_average": "7.241", "rank": "8", "rank_hist": {"w2025_22": "8"}, "popularity": "239.3911", "backdrop_path": "/cJvUJEEQ86LSjl4gFLkYpdCJC96.jpg", "release_date": "2025-04-09", "original_language": "en", "vote_count": "433", "until_date": "2025-05-27", "id": "1241436", "poster_path": "/fkVpNJugieKeTu7Se8uQRqRag2M.jpg", "video": false, "title": "Warfare. T

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 8:</b></font> Validación del API a través del uso de IPWidgets.

Para validar que todo funciona correctamente, hemos incluido una serie de elementos visuales (widgets) que permiten realizar consultas al API previamente desplegado. Para ello, únicamente debemos incluir en la siguiente celda la URL del API desplegado, incluyendo la etapa/stage, la cual habremos usado en validaciones preliminares.

Se deben realizar las siguientes acciones:

- Verificad el funcionamiento de los widgets usando la URL ya proporcionada (solución)
- Modificad la celda posterior para incluir la URL del API desplegado previamente
- Verificad el funcionamiento de los widgets

In [1]:
import os
#BASE_URL = os.environ.get('API_GATEWAY_SOLUTION_URL', '<INSTRUCTOR_API_URL>')  # URL SOLUCIÓN --> PARA VALIDAR
BASE_URL = os.environ.get('API_GATEWAY_URL', '<YOUR_API_GATEWAY_URL>')

<div class="alert alert-block alert-warning"> 
 La siguiente celda no debe ser modificada, sólo ejecutada (usará el valor de BASE_URL definido previamente)
</div>

In [2]:
%matplotlib inline

import requests
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from matplotlib.ticker import MaxNLocator    
    
def display_movie_info(year, month):    
    url = BASE_URL + f'/list/{year}/{month}'
    res = requests.get(url)
    data = res.json()
    
    dropdown_movies = widgets.Dropdown(
        options=[(x['title'], x['id']) for x in data],    
        description='Pelicula'
    )
    display(dropdown_movies)
    movie_data = widgets.interactive_output(display_movie, {'_id': dropdown_movies})
    display(movie_data)
    
def display_movie(_id):
    url = BASE_URL + f'/movies/{_id}'
    res = requests.get(url)
    data = res.json()
    if data:
        output1 = widgets.Output()
        with output1:
            poster_path = 'http://image.tmdb.org/t/p/w185'+data['poster_path']
            html = HTML(f'''
                <p>
                    <b>Estreno:</b> {data["release_date"]}</br>
                    <b>Hasta:</b> {data["until_date"]}</br>
                    <b>Votos:</b> {data["vote_count"]}</br>                    
                    <b>Media:</b> {data["vote_average"]}
                </p>
                <img src="{poster_path}" style=max-width:185px;"/>
            ''')
            display(html)

        output2 = widgets.Output()
        with output2:
            ranks = [int(value) for value in data['rank_hist'].values()]
            weeks = list(data['rank_hist'].keys())

            # Configurar grafica
            fig = plt.figure(figsize=(8, 5), dpi=100)
            ax = fig.gca()
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            plt.xticks(range(len(weeks)), weeks)
            plt.ylim(0, max(ranks) + 1)

            plt.title("Ranking semanal")
            plt.xlabel("Semana")
            plt.ylabel("Posición")

            # Dibujar grafica
            plt.plot(ranks, marker='o', linewidth=2) 
            plt.show()

        two_columns = widgets.HBox([output1, output2])
        display(two_columns)


dropdown_year = widgets.Dropdown(
    options=[2025],    
    description='Año'
)

dropdown_month = widgets.Dropdown(
    options=range(1, 13),    
    description='Mes'
)
movie_info = widgets.interactive_output(display_movie_info, {'year': dropdown_year, 'month': dropdown_month})

display(dropdown_year)
display(dropdown_month)
display(movie_info)

Dropdown(description='Año', options=(2025,), value=2025)

Dropdown(description='Mes', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12), value=1)

Output()

<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> EJERCICIO 9 (OPCIONAL):</b></font> Monitorización a través de Cloudwatch.

Si se desea profundizar/investigar sobre el uso de Cloudwatch, se propone un ejercicio opcional a tal efecto.

AWS Cloudwatch es un servicio de monitorización que permite crear paneles/dashboards mostrando datos sobre el funcionamiento de otros servicios. Por ejemplo, podemos mostrar las invocaciones totales a las funciones lambda o las capacidades de lectura de una tabla de DynamoDB.

El ejercicio propone la generación de un panel que muestre datos, que desde vuestro punto de vista, sean relevantes para la monitorización de la aplicación en su conjunto. Este ejercicio se realizará totalmente a través de la consola de AWS.


<font color="#00586D" size=3><b><i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> VALIDACIÓN</b></font>

- Generad una imagen con un pantallazo que muestre el panel de Cloudwatch desarrollado, y guardadla en la ruta (junto al notebook) `/img/eje_9.png`

<img src="img/eje_9.png" alt="Validacion 9">

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#113D68"></i>
 </font></div>

<div style="text-align: right">
<a href="#inicio"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#00586D"></i> </font></div>